# CNN para Clasificacion de Ordenes de Insectos
## Inteligencia Artificial II - Transfer Learning con EfficientNetB0

### Descripcion del Proyecto

Red Neuronal Convolucional para clasificar imagenes de insectos en sus ordenes taxonomicos
usando Transfer Learning con EfficientNetB0 preentrenada en ImageNet.

**Dataset:** iNaturalist - Ordenes de Insectos (train / val / test)

**Tecnicas aplicadas:**
- Transfer Learning + Fine-Tuning progresivo
- Regularizacion: Dropout, BatchNorm, L2
- Optimizacion: Adam, ReduceLROnPlateau, EarlyStopping
- Manejo de desbalance: class_weight + Data Augmentation
- Analisis: Matriz de Confusion, ROC, Grad-CAM


## Seccion 1 - Configuracion e Importaciones

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from collections import Counter

import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import label_binarize
from matplotlib.patches import Patch

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#e6edf3',
    'text.color':       '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'grid.color':       '#21262d',
    'axes.titlesize':   14,
    'axes.labelsize':   12,
})

PALETTE = [
    '#58a6ff', '#3fb950', '#d2a8ff', '#ff7b72', '#ffa657',
    '#79c0ff', '#56d364', '#bc8cff', '#ff9090', '#ffb86c',
    '#1f6feb', '#2ea043', '#8957e5', '#da3633', '#e3b341'
]

print('TensorFlow:', tf.__version__)


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    [tf.config.experimental.set_memory_growth(g, True) for g in gpus]
    print('GPU detectada:', [g.name for g in gpus])
else:
    print('Sin GPU - usando CPU')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATASET_ROOT   = Path(r'C:\\Users\\lucia\\Desktop\\CNN dataset\\dataset_organizado')
TRAIN_DIR      = DATASET_ROOT / 'train'
VAL_DIR        = DATASET_ROOT / 'val'
TEST_DIR       = DATASET_ROOT / 'test'
SAVE_DIR       = Path(r'C:\\Users\\lucia\\Desktop\\IA 2\\CNN')
SAVE_DIR.mkdir(exist_ok=True)

IMG_SIZE       = (224, 224)
BATCH_SIZE     = 32
EPOCHS_P1      = 20
EPOCHS_P2      = 30
LR_P1          = 1e-3
LR_P2          = 1e-5
MIN_TRAIN_IMGS = 3

print('Dataset:', DATASET_ROOT)
print('Imagen size:', IMG_SIZE, '| Batch:', BATCH_SIZE)


## Seccion 2 - Analisis Exploratorio del Dataset (EDA)

Antes de entrenar cualquier modelo se analiza la naturaleza del dataset:
distribucion de clases, nivel de desbalance y aspecto visual de las imagenes.


In [ ]:
def count_images(split_dir):
    exts = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
    counts = {}
    for cls_dir in sorted(split_dir.iterdir()):
        if cls_dir.is_dir():
            n = sum(len(list(cls_dir.glob(e))) for e in exts)
            counts[cls_dir.name] = n
    return counts

train_cnt = count_images(TRAIN_DIR)
val_cnt   = count_images(VAL_DIR)
test_cnt  = count_images(TEST_DIR)

df = pd.DataFrame({
    'Orden':      list(train_cnt.keys()),
    'Train':      list(train_cnt.values()),
    'Validacion': [val_cnt.get(k, 0)  for k in train_cnt],
    'Test':       [test_cnt.get(k, 0) for k in train_cnt],
})
df['Total'] = df[['Train', 'Validacion', 'Test']].sum(axis=1)
df['Pct']   = (df['Total'] / df['Total'].sum() * 100).round(2)
df = df.sort_values('Train', ascending=False).reset_index(drop=True)

print(df.to_string(index=False))
print(f"\nTotal train: {df['Train'].sum():,}  |  val: {df['Validacion'].sum():,}  |  test: {df['Test'].sum():,}")


In [ ]:
df_valid  = df[df['Train'] >= MIN_TRAIN_IMGS].copy()
CLASSES   = df_valid['Orden'].tolist()
N_CLASSES = len(CLASSES)
excluded  = df[df['Train'] < MIN_TRAIN_IMGS]['Orden'].tolist()

print('Clases validas  (', N_CLASSES, '):', CLASSES)
print('Clases excluidas(', len(excluded), '):', excluded)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Distribucion del Dataset - Ordenes de Insectos', fontsize=15, fontweight='bold')

df_p = df_valid.sort_values('Train', ascending=True)

for ax, scale, subtitle in zip(
        axes,
        ['linear', 'log'],
        ['Escala Lineal', 'Escala Logaritmica (evidencia el desbalance)']):
    bars = ax.barh(df_p['Orden'], df_p['Train'],
                   color=PALETTE[:len(df_p)], edgecolor='none', height=0.7, alpha=0.9)
    ax.set_xscale(scale)
    if scale == 'linear':
        for b, v in zip(bars, df_p['Train']):
            ax.text(b.get_width() + df_p['Train'].max() * 0.01,
                    b.get_y() + b.get_height() / 2,
                    f'{v:,}', va='center', fontsize=8.5, fontweight='bold')
        ax.set_xlim(0, df_p['Train'].max() * 1.18)
    ax.set_xlabel('Imagenes de Entrenamiento')
    ax.set_title(subtitle, fontsize=12)
    ax.grid(axis='x', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

ratio = df_valid['Train'].max() // max(df_valid['Train'].min(), 1)
print('Ratio desbalance max/min:', ratio, ':1')
plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig01_distribucion_clases.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
df_p2 = df_valid.sort_values('Train', ascending=False)
x = np.arange(len(df_p2))
w = 0.27

ax.bar(x - w, df_p2['Train'],      width=w, label='Train',      color='#58a6ff', alpha=0.9)
ax.bar(x,     df_p2['Validacion'], width=w, label='Validacion',  color='#3fb950', alpha=0.9)
ax.bar(x + w, df_p2['Test'],       width=w, label='Test',        color='#d2a8ff', alpha=0.9)

ax.set_yscale('log')
ax.set_xticks(x)
ax.set_xticklabels(df_p2['Orden'], rotation=42, ha='right', fontsize=10)
ax.set_ylabel('Imagenes (escala log)')
ax.set_title('Distribucion Train / Validacion / Test por Orden', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig02_train_val_test.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
def load_sample(cls_dir, idx=0):
    imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
    if not imgs:
        return None
    img = cv2.imread(str(imgs[min(idx, len(imgs) - 1)]))
    if img is None:
        return None
    return cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (180, 180))

n_cols = 5
n_rows = (N_CLASSES + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
fig.suptitle('Galeria de Muestras - Un ejemplo por Orden', fontsize=14, fontweight='bold')
flat = axes.flatten()

for i, cls in enumerate(CLASSES):
    img = load_sample(TRAIN_DIR / cls)
    c = PALETTE[i % len(PALETTE)]
    if img is not None:
        flat[i].imshow(img)
    flat[i].set_title(f'{cls}\n{train_cnt.get(cls, 0):,} imgs', fontsize=9, color=c, fontweight='bold')
    flat[i].axis('off')

for j in range(i + 1, len(flat)):
    flat[j].axis('off')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig03_galeria_clases.png', dpi=150, bbox_inches='tight')
plt.show()


## Seccion 3 - Preprocesamiento y Data Augmentation

**Transformaciones aplicadas durante el entrenamiento:**

| Tecnica       | Valor    | Proposito                              |
|---------------|----------|----------------------------------------|
| Rotacion      | +/-30    | Orientacion variable del insecto       |
| Shift H/V     | 20%      | Posicion en el encuadre                |
| Zoom          | 20%      | Distancia de la fotografia             |
| Flip H        | 50%      | Simetria bilateral                     |
| Brillo        | +/-20%   | Variaciones de iluminacion             |
| Shear         | 15%      | Perspectiva                            |

Para validacion y test se aplica solo el preprocesamiento de EfficientNetB0 (sin augmentation).


In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    rotation_range         = 30,
    width_shift_range      = 0.2,
    height_shift_range     = 0.2,
    shear_range            = 0.15,
    zoom_range             = 0.2,
    horizontal_flip        = True,
    brightness_range       = [0.8, 1.2],
    fill_mode              = 'nearest'
)

val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

flow_kwargs = dict(target_size=IMG_SIZE, batch_size=BATCH_SIZE,
                   class_mode='categorical', classes=CLASSES)

train_gen = train_datagen.flow_from_directory(TRAIN_DIR,    shuffle=True,  seed=SEED, **flow_kwargs)
val_gen   = val_test_datagen.flow_from_directory(VAL_DIR,   shuffle=False, **flow_kwargs)
test_gen  = val_test_datagen.flow_from_directory(TEST_DIR,  shuffle=False, **flow_kwargs)

CLASS_NAMES = list(train_gen.class_indices.keys())
print('Clases:', CLASS_NAMES)
print('Steps -> train:', len(train_gen), '| val:', len(val_gen), '| test:', len(test_gen))


In [ ]:
y_labels = []
for i, cls in enumerate(CLASS_NAMES):
    exts = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
    n = sum(len(list((TRAIN_DIR / cls).glob(e))) for e in exts)
    y_labels.extend([i] * n)
y_labels = np.array(y_labels)

cw_arr  = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=y_labels)
CW_DICT = {i: w for i, w in enumerate(cw_arr)}

print('Class Weights:')
for i, cls in enumerate(CLASS_NAMES):
    print(f'  {cls:<22}: {cw_arr[i]:8.3f}')


In [ ]:
sample_cls = next((c for c in ['Lepidoptera', 'Hymenoptera', 'Coleoptera'] if c in CLASSES), CLASSES[0])
imgs = list((TRAIN_DIR / sample_cls).glob('*.jpg'))

if imgs:
    orig = cv2.cvtColor(cv2.imread(str(imgs[0])), cv2.COLOR_BGR2RGB)
    orig = cv2.resize(orig, IMG_SIZE)
    arr  = np.expand_dims(orig.astype(np.float32), 0)

    aug_gen = ImageDataGenerator(
        rotation_range=30, width_shift_range=0.2, height_shift_range=0.2,
        shear_range=0.15, zoom_range=0.2, horizontal_flip=True,
        brightness_range=[0.8, 1.2], fill_mode='nearest')
    it = aug_gen.flow(arr, batch_size=1, seed=SEED)

    fig, axes = plt.subplots(2, 6, figsize=(18, 7))
    fig.suptitle(f'Data Augmentation - Clase: {sample_cls}', fontsize=13, fontweight='bold')

    axes[0, 0].imshow(orig)
    axes[0, 0].set_title('Original', fontsize=10, fontweight='bold')
    axes[0, 0].axis('off')

    for idx in range(1, 12):
        r, c = divmod(idx, 6)
        aug = next(it)[0].clip(0, 255).astype(np.uint8)
        axes[r, c].imshow(aug)
        axes[r, c].set_title(f'Aug {idx}', fontsize=9)
        axes[r, c].axis('off')

    plt.tight_layout()
    plt.savefig(SAVE_DIR / 'fig04_augmentation.png', dpi=150, bbox_inches='tight')
    plt.show()


## Seccion 4 - Arquitectura del Modelo CNN

### Por que EfficientNetB0?

- Parametros: ~5.3M (muy eficiente)
- Top-1 Accuracy en ImageNet: 77.1%
- Compound Scaling: optimiza ancho, profundidad y resolucion a la vez
- Mejor rendimiento/costo vs ResNet o VGG

### Estrategia de entrenamiento en 2 fases

**Fase 1 - Feature Extraction:**
EfficientNetB0 congelado -> GlobalAvgPool -> BN -> Dense(512) -> Dense(256) -> Softmax
LR = 1e-3, solo la cabeza personalizada se actualiza.

**Fase 2 - Fine-Tuning:**
Top 30% de EfficientNetB0 descongelado, LR = 1e-5.

### Regularizacion en la cabeza
- Dropout decreciente: 0.4 -> 0.3 -> 0.2
- BatchNormalization despues de GlobalAvgPool y Dense
- L2 (lambda=1e-4) en capas Dense


In [ ]:
def build_model(n_classes, lr=1e-3):
    base = EfficientNetB0(include_top=False, weights='imagenet', input_shape=(*IMG_SIZE, 3))
    base.trainable = False

    inp = keras.Input(shape=(*IMG_SIZE, 3), name='imagen')
    x = base(inp, training=False)

    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.Dropout(0.4, name='drop1')(x)
    x = layers.Dense(512, activation='relu', name='dense1',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.Dropout(0.3, name='drop2')(x)
    x = layers.Dense(256, activation='relu', name='dense2',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.2, name='drop3')(x)
    out = layers.Dense(n_classes, activation='softmax', name='prediccion')(x)

    m = keras.Model(inp, out, name='CNN_Insectos_EfficientNetB0')
    m.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
    )
    return m, base

model, base_model = build_model(N_CLASSES, LR_P1)
total   = model.count_params()
train_p = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'Parametros totales:     {total:,}')
print(f'Parametros entrenables: {train_p:,}  ({train_p/total*100:.1f}%)')


In [ ]:
model.summary()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 14))
ax.set_xlim(0, 10)
ax.set_ylim(0, 22)
ax.axis('off')
fig.patch.set_facecolor('#0d1117')

arch = [
    ('Input Image',                   '224 x 224 x 3',     '#1f6feb', 21),
    ('EfficientNetB0 (FROZEN)',        '7 x 7 x 1280',      '#8957e5', 18),
    ('GlobalAveragePooling2D',         '1280',               '#2ea043', 15.2),
    ('BatchNormalization',             '1280',               '#e3b341', 13.2),
    ('Dropout (0.4)',                  '1280',               '#da3633', 12.0),
    ('Dense (512, relu)  L2=1e-4',    '512',                '#58a6ff', 10.5),
    ('BatchNorm + Dropout (0.3)',      '512',                '#e3b341', 9.0),
    ('Dense (256, relu)  L2=1e-4',    '256',                '#58a6ff', 7.5),
    ('Dropout (0.2)',                  '256',                '#da3633', 6.3),
    (f'Dense ({N_CLASSES}, softmax)', f'{N_CLASSES} clases', '#ff7b72', 4.5),
    ('Clasificacion Final',            'Orden de Insecto',   '#3fb950', 2.8),
]

ys = [a[3] for a in arch]
for i in range(len(ys) - 1):
    ax.annotate('', xy=(5, ys[i+1] + 0.5), xytext=(5, ys[i] - 0.5),
                arrowprops=dict(arrowstyle='->', color='#8b949e', lw=1.8))

for name, shape, color, y in arch:
    rect = plt.Rectangle((1.2, y - 0.5), 7.6, 1.0,
                          facecolor=color, alpha=0.18,
                          edgecolor=color, lw=2)
    ax.add_patch(rect)
    ax.text(5, y, name, ha='center', va='center', fontsize=10, fontweight='bold', color=color)
    ax.text(9.0, y, shape, ha='right', va='center', fontsize=8.5, color='#8b949e', style='italic')

ax.set_title('Arquitectura CNN - EfficientNetB0 + Cabeza Personalizada',
             fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig05_arquitectura.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()


## Seccion 5 - Entrenamiento Fase 1: Feature Extraction

La base de EfficientNetB0 esta congelada. Solo se actualizan las capas Dense de la cabeza.

| Callback          | Configuracion                  | Proposito                    |
|-------------------|-------------------------------|------------------------------|
| EarlyStopping     | paciencia=8, val_accuracy     | Detiene si no mejora         |
| ReduceLROnPlateau | factor=0.5, paciencia=4       | Reduce LR si val_loss estanca|
| ModelCheckpoint   | save_best_only=True           | Guarda el mejor modelo       |


In [ ]:
cb_p1 = [
    EarlyStopping(monitor='val_accuracy', patience=8,
                  restore_best_weights=True, verbose=1, mode='max'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4,
                      min_lr=1e-7, verbose=1),
    ModelCheckpoint(str(SAVE_DIR / 'modelo_fase1_best.h5'),
                    monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
]

history_p1 = model.fit(
    train_gen,
    epochs          = EPOCHS_P1,
    validation_data = val_gen,
    class_weight    = CW_DICT,
    callbacks       = cb_p1,
    verbose         = 1
)

print('Mejor val_accuracy Fase 1:', max(history_p1.history['val_accuracy']))


In [ ]:
def plot_curves(hist, title='', save='curvas.png'):
    acc      = hist.history['accuracy']
    val_acc  = hist.history['val_accuracy']
    loss     = hist.history['loss']
    val_loss = hist.history['val_loss']
    ep       = range(1, len(acc) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Curvas de Entrenamiento - ' + title, fontsize=14, fontweight='bold')

    ax1 = axes[0]
    ax1.plot(ep, acc,     color='#58a6ff', lw=2.5, label='Train Accuracy')
    ax1.plot(ep, val_acc, color='#3fb950', lw=2.5, linestyle='--', label='Val Accuracy')
    best_e = np.argmax(val_acc) + 1
    ax1.axvline(best_e, color='#d2a8ff', linestyle=':', lw=1.5)
    ax1.scatter(best_e, max(val_acc), color='#d2a8ff', s=90, zorder=5,
                label=f'Mejor epoca {best_e}: {max(val_acc):.4f}')
    ax1.set(xlabel='Epoca', ylabel='Accuracy', ylim=(0, 1))
    ax1.set_title('Accuracy vs Epoca')
    ax1.legend(fontsize=10)
    ax1.grid(alpha=0.3)
    ax1.spines[['top', 'right']].set_visible(False)

    ax2 = axes[1]
    ax2.plot(ep, loss,     color='#ff7b72', lw=2.5, label='Train Loss')
    ax2.plot(ep, val_loss, color='#ffa657', lw=2.5, linestyle='--', label='Val Loss')
    best_le = np.argmin(val_loss) + 1
    ax2.axvline(best_le, color='#d2a8ff', linestyle=':', lw=1.5)
    ax2.scatter(best_le, min(val_loss), color='#d2a8ff', s=90, zorder=5,
                label=f'Mejor epoca {best_le}: {min(val_loss):.4f}')
    ax2.set(xlabel='Epoca', ylabel='Loss')
    ax2.set_title('Loss vs Epoca')
    ax2.legend(fontsize=10)
    ax2.grid(alpha=0.3)
    ax2.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.savefig(SAVE_DIR / save, dpi=150, bbox_inches='tight')
    plt.show()

plot_curves(history_p1, 'Fase 1 - Feature Extraction', 'fig06_curvas_fase1.png')


## Seccion 6 - Fine-Tuning: Fase 2

Se descongelan las ultimas capas de EfficientNetB0 (top 30%) para que se adapten
al dominio de insectos. Se usa un learning rate muy pequeno (1e-5) para no destruir
los pesos preentrenados.

```
Capas congeladas : 0% - 70%   (caracteristicas base: bordes, texturas)
Capas libres     : 70% - 100% (caracteristicas de alto nivel)
```


In [ ]:
total_layers = len(base_model.layers)
fine_tune_at = int(total_layers * 0.70)
base_model.trainable = True

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

libre     = sum(1 for l in base_model.layers if l.trainable)
congelado = total_layers - libre
print(f'Capas totales EfficientNetB0: {total_layers}')
print(f'Capas congeladas : {congelado} (~70%)')
print(f'Capas libres     : {libre} (~30%)')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_P2),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

train_p2 = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'Parametros entrenables ahora: {train_p2:,}')


In [ ]:
cb_p2 = [
    EarlyStopping(monitor='val_accuracy', patience=10,
                  restore_best_weights=True, verbose=1, mode='max'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5,
                      min_lr=1e-9, verbose=1),
    ModelCheckpoint(str(SAVE_DIR / 'modelo_final_best.h5'),
                    monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
]

history_p2 = model.fit(
    train_gen,
    epochs          = EPOCHS_P2,
    validation_data = val_gen,
    class_weight    = CW_DICT,
    callbacks       = cb_p2,
    verbose         = 1
)

print('Mejor val_accuracy Fine-Tuning:', max(history_p2.history['val_accuracy']))


In [ ]:
plot_curves(history_p2, 'Fase 2 - Fine-Tuning', 'fig07_curvas_fase2.png')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Historial Completo - Fase 1 + Fine-Tuning', fontsize=14, fontweight='bold')

sep = len(history_p1.history['accuracy'])

for i, (metric, colors, ylabel) in enumerate([
    ('accuracy', ('#58a6ff', '#3fb950'), 'Accuracy'),
    ('loss',     ('#ff7b72', '#ffa657'), 'Loss'),
]):
    tr_all  = history_p1.history[metric]         + history_p2.history[metric]
    val_all = history_p1.history[f'val_{metric}'] + history_p2.history[f'val_{metric}']
    ep_all  = range(1, len(tr_all) + 1)
    ax = axes[i]
    ax.plot(ep_all, tr_all,  color=colors[0], lw=2.5, label=f'Train {ylabel}')
    ax.plot(ep_all, val_all, color=colors[1], lw=2.5, linestyle='--', label=f'Val {ylabel}')
    ax.axvline(sep, color='#d2a8ff', linestyle=':', lw=2)
    ax.text(sep + 0.3, ax.get_ylim()[1] * 0.95,
            'Fase 1 | Fine-Tuning', color='#d2a8ff', fontsize=8.5)
    ax.set(xlabel='Epoca', ylabel=ylabel)
    ax.set_title(f'{ylabel} - Historial Completo')
    if i == 0:
        ax.set_ylim(0, 1)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig08_historial_completo.png', dpi=150, bbox_inches='tight')
plt.show()

model.save(str(SAVE_DIR / 'modelo_cnn_insectos_final.h5'))
print('Modelo guardado: modelo_cnn_insectos_final.h5')


## Seccion 7 - Evaluacion Completa del Modelo

1. Predicciones en el set de test
2. Metricas globales: Accuracy, Top-3 Accuracy
3. Reporte de clasificacion por clase
4. Matriz de Confusion (absoluta y normalizada)
5. Recall por clase (diagonal de la matriz)
6. Curvas ROC (One-vs-Rest) + AUC
7. Analisis de errores con imagenes mal clasificadas


In [ ]:
print('Prediciendo en conjunto de test...')
test_gen.reset()
y_pred_prob = model.predict(test_gen, verbose=1)
y_pred      = np.argmax(y_pred_prob, axis=1)
y_true      = test_gen.classes[:len(y_pred)]
filenames   = np.array(test_gen.filenames)[:len(y_pred)]

test_loss, test_acc, test_top3 = model.evaluate(test_gen, verbose=0)
print(f'Loss          : {test_loss:.4f}')
print(f'Accuracy      : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print(f'Top-3 Accuracy: {test_top3:.4f}  ({test_top3*100:.2f}%)')


In [ ]:
report_str  = classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0)
report_dict = classification_report(y_true, y_pred, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
print(report_str)
df_rep = pd.DataFrame(report_dict).T.iloc[:-3]


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle('Metricas por Clase - Conjunto de Test', fontsize=14, fontweight='bold')

for ax, (metric, color, label) in zip(axes, [
    ('precision', '#58a6ff', 'Precision'),
    ('recall',    '#3fb950', 'Recall'),
    ('f1-score',  '#d2a8ff', 'F1-Score'),
]):
    vals = df_rep[metric].astype(float).values
    clss = df_rep.index.tolist()
    bars = ax.barh(clss, vals, color=color, alpha=0.85, edgecolor='none', height=0.7)
    ax.axvline(0.5, color='#ff7b72', linestyle='--', lw=1.5, alpha=0.7, label='0.5')
    ax.axvline(0.8, color='#ffa657', linestyle=':',  lw=1.2, alpha=0.6, label='0.8')
    for b, v in zip(bars, vals):
        ax.text(min(v + 0.02, 0.95), b.get_y() + b.get_height() / 2,
                f'{v:.2f}', va='center', fontsize=8.5, fontweight='bold')
    ax.set(xlim=(0, 1.12), xlabel=label, title=label)
    ax.legend(fontsize=8)
    ax.grid(axis='x', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig09_metricas_por_clase.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-10)

fig, axes = plt.subplots(1, 2, figsize=(22, 10))
fig.suptitle('Matriz de Confusion - Conjunto de Test', fontsize=15, fontweight='bold', y=1.01)

configs = [
    (cm,      'Blues',  'Conteo Absoluto',
     lambda v: str(int(v)) if v > 0 else ''),
    (cm_norm, 'YlOrRd', 'Normalizada por Fila (Recall por clase)',
     lambda v: f'{v:.2f}' if v > 0.01 else ''),
]

for ax, (data, cmap, title, fmt_fn) in zip(axes, configs):
    im = ax.imshow(data, cmap=cmap, aspect='auto',
                   vmin=0, vmax=(None if cmap == 'Blues' else 1))
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=8.5)
    ax.set_yticklabels(CLASS_NAMES, fontsize=8.5)
    ax.set_xlabel('Prediccion', fontsize=12, fontweight='bold')
    ax.set_ylabel('Etiqueta Real', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=12)
    thresh = data.max() / 2
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            txt = fmt_fn(data[i, j])
            if txt:
                ax.text(j, i, txt, ha='center', va='center', fontsize=7,
                        color='white' if data[i, j] > thresh else '#e6edf3',
                        fontweight='bold')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig10_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
diag   = cm_norm.diagonal()
df_diag = pd.DataFrame({'Clase': CLASS_NAMES, 'Recall': diag,
                         'N_Test': cm.sum(axis=1)}).sort_values('Recall', ascending=False)

colors_d = ['#3fb950' if v >= 0.7 else '#ffa657' if v >= 0.4 else '#ff7b72'
            for v in df_diag['Recall']]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(df_diag['Clase'], df_diag['Recall'], color=colors_d, edgecolor='none', height=0.7)
ax.axvline(0.7, color='#3fb950', linestyle='--', lw=1.5)
ax.axvline(0.4, color='#ff7b72', linestyle='--', lw=1.5)
legend_elems = [Patch(facecolor='#3fb950', label='Recall >= 70%'),
                Patch(facecolor='#ffa657', label='Recall 40-70%'),
                Patch(facecolor='#ff7b72', label='Recall < 40%')]
ax.legend(handles=legend_elems, fontsize=10)
for b, v, n in zip(bars, df_diag['Recall'], df_diag['N_Test']):
    ax.text(min(v + 0.01, 0.96), b.get_y() + b.get_height() / 2,
            f'{v:.3f}  ({int(n)} imgs)', va='center', fontsize=9)
ax.set(xlim=(0, 1.15), xlabel='Recall', title='Recall por Clase - Diagonal de la Matriz de Confusion')
ax.grid(axis='x', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig11_recall_por_clase.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
y_true_bin = label_binarize(y_true, classes=list(range(N_CLASSES)))

n_cols = 4
n_rows = (N_CLASSES + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 5))
fig.suptitle('Curvas ROC por Clase (One-vs-Rest)', fontsize=14, fontweight='bold', y=1.01)

auc_vals = []
flat_ax  = axes.flatten()

for i, cls in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_prob[:, i])
    roc_auc = auc(fpr, tpr)
    auc_vals.append(roc_auc)
    color = PALETTE[i % len(PALETTE)]
    ax = flat_ax[i]
    ax.plot(fpr, tpr, color=color, lw=2.5, label=f'AUC = {roc_auc:.3f}')
    ax.plot([0, 1], [0, 1], color='#8b949e', linestyle='--', lw=1.2)
    ax.fill_between(fpr, tpr, alpha=0.12, color=color)
    ax.set(xlim=[0, 1], ylim=[0, 1.05], xlabel='FPR', ylabel='TPR')
    ax.set_title(cls, fontsize=10, fontweight='bold', color=color)
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

for j in range(i + 1, len(flat_ax)):
    flat_ax[j].axis('off')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig12_curvas_roc.png', dpi=150, bbox_inches='tight')
plt.show()

print('AUC-ROC por clase:')
for cls, av in sorted(zip(CLASS_NAMES, auc_vals), key=lambda x: -x[1]):
    print(f'  {cls:<22}: {av:.4f}')
print(f'Promedio AUC-ROC: {np.mean(auc_vals):.4f}')


In [ ]:
auc_df = pd.DataFrame({'Clase': CLASS_NAMES, 'AUC': auc_vals}).sort_values('AUC', ascending=True)
colors_auc = ['#3fb950' if v >= 0.90 else '#ffa657' if v >= 0.75 else '#ff7b72'
              for v in auc_df['AUC']]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(auc_df['Clase'], auc_df['AUC'], color=colors_auc, edgecolor='none', height=0.7)
ax.axvline(0.90, color='#3fb950', linestyle='--', lw=1.5)
ax.axvline(0.75, color='#ffa657', linestyle='--', lw=1.5)
ax.axvline(0.5,  color='#ff7b72', linestyle=':',  lw=1.2)
for b, v in zip(bars, auc_df['AUC']):
    ax.text(v + 0.003, b.get_y() + b.get_height() / 2,
            f'{v:.4f}', va='center', fontsize=10, fontweight='bold')
legend_auc = [Patch(facecolor='#3fb950', label='AUC >= 0.90'),
              Patch(facecolor='#ffa657', label='AUC >= 0.75'),
              Patch(facecolor='#ff7b72', label='AUC < 0.75')]
ax.legend(handles=legend_auc, fontsize=10)
ax.set(xlim=(0.4, 1.06), xlabel='AUC-ROC',
       title=f'AUC-ROC por Clase - Promedio: {np.mean(auc_vals):.4f}')
ax.grid(axis='x', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig13_auc_resumen.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
mask_wrong = y_pred != y_true
wrong_idx  = np.where(mask_wrong)[0]

error_pairs   = [(CLASS_NAMES[y_true[i]], CLASS_NAMES[y_pred[i]]) for i in wrong_idx]
error_counter = Counter(error_pairs)

print(f'Errores: {len(wrong_idx)} / {len(y_true)} ({len(wrong_idx)/len(y_true)*100:.1f}%)')
print('\nTop-10 confusiones frecuentes:')
for (real, pred), cnt in error_counter.most_common(10):
    print(f'  {real:<22} -> {pred:<22}: {cnt}')

n_show  = min(12, len(wrong_idx))
sample_w = np.random.choice(wrong_idx, size=n_show, replace=False)
n_cols2  = 4
n_rows2  = (n_show + n_cols2 - 1) // n_cols2

fig, axes2 = plt.subplots(n_rows2, n_cols2, figsize=(18, n_rows2 * 5))
fig.suptitle('Imagenes Mal Clasificadas - Analisis de Errores', fontsize=13, fontweight='bold')
flat2 = axes2.flatten()

for idx, wi in enumerate(sample_w):
    ax = flat2[idx]
    fp  = TEST_DIR / filenames[wi]
    img = cv2.imread(str(fp))
    if img is not None:
        img = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (200, 200))
        ax.imshow(img)
    real = CLASS_NAMES[y_true[wi]]
    pred = CLASS_NAMES[y_pred[wi]]
    conf = y_pred_prob[wi, y_pred[wi]]
    ax.set_title(f'Real: {real}\nPred: {pred} ({conf:.1%})',
                 fontsize=9, color='#ff7b72', fontweight='bold')
    ax.axis('off')
    for s in ax.spines.values():
        s.set_edgecolor('#ff7b72'); s.set_linewidth(2); s.set_visible(True)

for j in range(idx + 1, len(flat2)):
    flat2[j].axis('off')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig14_errores.png', dpi=150, bbox_inches='tight')
plt.show()


## Seccion 8 - Interpretabilidad con Grad-CAM

Grad-CAM (Gradient-weighted Class Activation Mapping) visualiza que regiones
de la imagen el modelo considera mas importantes para su decision.

Interpretacion del mapa de calor:
- Rojo/Naranja: region muy relevante para la decision
- Amarillo: region de importancia media
- Azul: region poco relevante


In [ ]:
def make_gradcam(model, img_array, layer_name, pred_idx=None):
    grad_model = keras.Model(
        inputs  = model.inputs,
        outputs = [model.get_layer(layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        if pred_idx is None:
            pred_idx = tf.argmax(preds[0])
        score = preds[:, pred_idx]
    grads   = tape.gradient(score, conv_out)
    pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.reduce_sum(conv_out[0] * pooled, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_gradcam(orig_rgb, heatmap):
    hmap = cv2.resize(heatmap, (orig_rgb.shape[1], orig_rgb.shape[0]))
    hmap_col = cv2.applyColorMap(np.uint8(255 * hmap), cv2.COLORMAP_JET)
    hmap_col = cv2.cvtColor(hmap_col, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(orig_rgb, 0.6, hmap_col, 0.4, 0), hmap_col

conv_layer = None
for layer in reversed(model.layers):
    if hasattr(layer, 'output') and len(layer.output.shape) == 4:
        conv_layer = layer.name
        break
print('Capa convolucional seleccionada:', conv_layer)


In [ ]:
gradcam_classes = [c for c in ['Diptera', 'Hymenoptera', 'Lepidoptera', 'Coleoptera'] if c in CLASS_NAMES][:4]

fig, axes = plt.subplots(len(gradcam_classes), 3, figsize=(15, len(gradcam_classes) * 5))
fig.suptitle('Grad-CAM - Regiones de Atencion del Modelo por Clase', fontsize=13, fontweight='bold')

if len(gradcam_classes) == 1:
    axes = [axes]

for row, cls_name in enumerate(gradcam_classes):
    cls_dir  = TEST_DIR / cls_name
    imgs_cls = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
    if not imgs_cls:
        for ax in axes[row]:
            ax.axis('off')
        continue

    orig = cv2.cvtColor(cv2.imread(str(imgs_cls[0])), cv2.COLOR_BGR2RGB)
    orig = cv2.resize(orig, IMG_SIZE)
    inp  = preprocess_input(np.expand_dims(orig.astype(np.float32), 0))

    axes[row][0].imshow(orig)
    axes[row][0].set_title(f'{cls_name} - Original', fontsize=11, fontweight='bold',
                           color=PALETTE[row % len(PALETTE)])
    axes[row][0].axis('off')

    try:
        cls_idx = CLASS_NAMES.index(cls_name)
        heatmap = make_gradcam(model, inp, conv_layer, cls_idx)
        overlay, hmap_col = overlay_gradcam(orig, heatmap)

        axes[row][1].imshow(hmap_col)
        axes[row][1].set_title('Mapa de Calor Grad-CAM', fontsize=11)
        axes[row][1].axis('off')

        axes[row][2].imshow(overlay)
        axes[row][2].set_title('Superpuesto sobre Original', fontsize=11)
        axes[row][2].axis('off')
    except Exception as e:
        print(f'  {cls_name}: {e}')
        axes[row][1].axis('off')
        axes[row][2].axis('off')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig15_gradcam.png', dpi=150, bbox_inches='tight')
plt.show()


## Seccion 9 - Dashboard Final

In [ ]:
macro_f1    = report_dict['macro avg']['f1-score']
weighted_f1 = report_dict['weighted avg']['f1-score']
prec_w      = report_dict['weighted avg']['precision']
rec_w       = report_dict['weighted avg']['recall']
mean_auc    = float(np.mean(auc_vals))

fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor('#0d1117')
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.6, wspace=0.45)
fig.suptitle('Dashboard Final - CNN Clasificacion de Ordenes de Insectos',
             fontsize=15, fontweight='bold', color='#e6edf3')

CARDS = [
    (test_acc,    'Accuracy (Test)',      '#58a6ff'),
    (test_top3,   'Top-3 Accuracy',       '#3fb950'),
    (macro_f1,    'F1-Score Macro',       '#d2a8ff'),
    (weighted_f1, 'F1-Score Weighted',    '#ffa657'),
    (prec_w,      'Precision Weighted',   '#79c0ff'),
    (rec_w,       'Recall Weighted',      '#56d364'),
    (mean_auc,    'AUC-ROC Promedio',     '#ff7b72'),
    (N_CLASSES,   'Clases',               '#e3b341'),
]

for idx, (val, label, color) in enumerate(CARDS):
    r, c = divmod(idx, 4)
    ax = fig.add_subplot(gs[r, c])
    ax.set_facecolor('#161b22')
    ax.axis('off')
    fmt = '.0f' if label == 'Clases' else '.4f'
    ax.text(0.5, 0.60, f'{val:{fmt}}', ha='center', va='center',
            fontsize=26, fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.22, label, ha='center', va='center',
            fontsize=10, color='#8b949e', transform=ax.transAxes)
    rect = plt.Rectangle((0, 0), 1, 1, fill=False, edgecolor=color, lw=2,
                          transform=ax.transAxes)
    ax.add_patch(rect)

plt.savefig(SAVE_DIR / 'fig16_dashboard_final.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()


In [ ]:
metrics_mat = np.array([
    [report_dict[cls]['precision'] for cls in CLASS_NAMES],
    [report_dict[cls]['recall']    for cls in CLASS_NAMES],
    [report_dict[cls]['f1-score']  for cls in CLASS_NAMES],
    auc_vals,
])

fig, ax = plt.subplots(figsize=(17, 5))
im = ax.imshow(metrics_mat, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)

ax.set_xticks(range(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES, rotation=42, ha='right', fontsize=9)
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(['Precision', 'Recall', 'F1-Score', 'AUC-ROC'], fontsize=11)
ax.set_title('Mapa de Calor de Metricas - Todas las Clases', fontsize=13, fontweight='bold')

for i in range(4):
    for j in range(len(CLASS_NAMES)):
        v = metrics_mat[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7.5,
                color='white' if v < 0.35 or v > 0.80 else '#0d1117',
                fontweight='bold')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig17_heatmap_metricas.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
generated = sorted(SAVE_DIR.glob('fig*.png')) + sorted(SAVE_DIR.glob('modelo*.h5'))
print('Archivos generados en:', SAVE_DIR)
for f in generated:
    kb = f.stat().st_size / 1024
    print(f'  {f.name:<48}  {kb:6.1f} KB')


## Seccion 10 - Conclusiones

### Resumen del Proyecto

Se implemento una CNN con EfficientNetB0 y Transfer Learning en 2 fases para clasificar
imagenes de insectos en sus ordenes taxonomicos.

### Tecnicas de Optimizacion

| Tecnica           | Configuracion              | Impacto                         |
|-------------------|---------------------------|---------------------------------|
| Adam Optimizer    | LR=1e-3 -> 1e-5           | Convergencia rapida y estable   |
| ReduceLROnPlateau | factor=0.5, paciencia=4   | Evita estancamiento de val_loss |
| EarlyStopping     | paciencia=8-10            | Previene overfitting            |
| Transfer Learning | EfficientNetB0 + ImageNet | Excelente baseline inicial      |
| Fine-Tuning       | Top 30% descongelado      | Adaptacion al dominio insectos  |

### Tecnicas de Regularizacion

| Tecnica           | Configuracion             | Impacto                                 |
|-------------------|---------------------------|-----------------------------------------|
| Dropout           | 0.4 -> 0.3 -> 0.2         | Reduce coadaptacion de neuronas         |
| BatchNormalization| Despues de GAP y Dense    | Estabiliza distribucion de activaciones |
| L2 Regularization | lambda=1e-4 en Dense      | Penaliza pesos excesivamente grandes    |
| Data Augmentation | Rotacion, zoom, flip, etc | Aumenta diversidad del entrenamiento    |
| Class Weights     | compute_class_weight      | Compensa el desbalance extremo          |

### Analisis del Desbalance

El dataset presenta desbalance extremo (Diptera ~70% de las imagenes). Los class weights
permiten que el modelo no ignore las clases minoritarias. Las clases con muy pocas
muestras siguen siendo las mas dificiles de clasificar correctamente.

### Interpretabilidad (Grad-CAM)

El analisis Grad-CAM confirma que el modelo focaliza en las caracteristicas morfologicas
correctas de los insectos (alas, abdomen, antenas) y no en el fondo de la imagen.

### Trabajo Futuro

- Oversampling para clases minoritarias (CutMix, MixUp)
- Modelos mas grandes: EfficientNetB3/B5 o Vision Transformer (ViT)
- Label smoothing para mejor calibracion
- Ensemble de multiples modelos
